# KMP 字符串匹配算法详解

## 算法简介

KMP（Knuth-Morris-Pratt）算法是一种改进的字符串匹配算法，由D.E.Knuth、J.H.Morris和V.R.Pratt同时发现，因此称为KMP算法。

### 核心思想

- **问题**：在文本串 `text` 中查找模式串 `pattern` 的位置
- **朴素算法**：时间复杂度 O(m*n)，每次不匹配就回退
- **KMP优化**：利用已经匹配的信息，避免不必要的回退，时间复杂度 O(m+n)

### 关键概念

1. **前缀**：字符串的前缀是从第一个字符开始的连续子串（不包含整个字符串）
2. **后缀**：字符串的后缀是到最后一个字符结束的连续子串（不包含整个字符串）
3. **next数组**：记录模式串中每个位置之前的子串的最长相等前后缀长度

### 本notebook包含的实现方法

1. 朴素字符串匹配算法（用于对比）
2. 标准KMP算法实现
3. 支持查找所有匹配的KMP算法
4. 面向对象的KMP实现
5. 简洁版KMP实现

## 方法1: 朴素字符串匹配算法（Brute Force）

### 算法思路

逐个位置尝试匹配，如果不匹配就将模式串向后移动一位，重新开始匹配。

### 时间复杂度

- **最坏情况**：O(m*n)，其中 m 是文本串长度，n 是模式串长度
- **空间复杂度**：O(1)

In [ ]:
def brute_force_search(text, pattern):
    """
    朴素字符串匹配算法
    
    参数:
        text: 文本串（主串）
        pattern: 模式串（要查找的子串）
    
    返回:
        匹配成功时返回第一个匹配位置的索引，失败返回-1
    """
    n = len(text)
    m = len(pattern)
    
    # 遍历文本串的每个可能的起始位置
    for i in range(n - m + 1):
        j = 0
        # 从当前位置开始逐个字符匹配
        while j < m and text[i + j] == pattern[j]:
            j += 1
        
        # 如果j等于模式串长度，说明完全匹配
        if j == m:
            return i
    
    return -1

# 测试用例
text1 = "ABABDABACDABABCABAB"
pattern1 = "ABABCABAB"

result = brute_force_search(text1, pattern1)
print(f"朴素算法测试:")
print(f"文本串: {text1}")
print(f"模式串: {pattern1}")
print(f"匹配位置: {result}")
if result != -1:
    print(f"匹配的子串: {text1[result:result+len(pattern1)]}")

## 方法2: 标准KMP算法实现

### 算法思路

1. **构建next数组**：预处理模式串，计算每个位置的最长相等前后缀长度
2. **模式匹配**：利用next数组跳过不必要的比较

### next数组的含义

- `next[i]` 表示模式串前 i 个字符组成的子串中，最长相等前后缀的长度
- 当匹配失败时，可以利用next数组确定模式串应该移动到的位置

### 时间复杂度

- **预处理**：O(n)
- **匹配**：O(m)
- **总体**：O(m+n)

In [ ]:
def get_next(pattern):
    """
    构建next数组（标准版本）
    
    参数:
        pattern: 模式串
    
    返回:
        next数组，next[i]表示pattern[0:i+1]的最长相等前后缀长度
    """
    m = len(pattern)
    next_arr = [0] * m  # 初始化next数组
    
    # next[0] = 0，单个字符没有真前后缀
    j = 0  # j表示当前最长相等前后缀的长度
    
    # 从第二个字符开始计算
    for i in range(1, m):
        # 如果不匹配，回退到前一个最长前缀的位置
        while j > 0 and pattern[i] != pattern[j]:
            j = next_arr[j - 1]
        
        # 如果匹配，前缀长度加1
        if pattern[i] == pattern[j]:
            j += 1
        
        next_arr[i] = j
    
    return next_arr


def kmp_search(text, pattern):
    """
    KMP字符串匹配算法
    
    参数:
        text: 文本串（主串）
        pattern: 模式串（要查找的子串）
    
    返回:
        匹配成功时返回第一个匹配位置的索引，失败返回-1
    """
    n = len(text)
    m = len(pattern)
    
    if m == 0:
        return 0
    
    # 构建next数组
    next_arr = get_next(pattern)
    
    j = 0  # 模式串的匹配位置
    
    # 遍历文本串
    for i in range(n):
        # 如果不匹配，根据next数组回退
        while j > 0 and text[i] != pattern[j]:
            j = next_arr[j - 1]
        
        # 如果匹配，模式串位置前进
        if text[i] == pattern[j]:
            j += 1
        
        # 如果完全匹配，返回匹配位置
        if j == m:
            return i - m + 1
    
    return -1


# 测试用例
print("标准KMP算法测试:")
print(f"文本串: {text1}")
print(f"模式串: {pattern1}")

# 显示next数组
next_array = get_next(pattern1)
print(f"\nnext数组: {next_array}")
print("next数组含义:")
for i, val in enumerate(next_array):
    print(f"  pattern[0:{i+1}] = '{pattern1[:i+1]}' 的最长相等前后缀长度: {val}")

result = kmp_search(text1, pattern1)
print(f"\n匹配位置: {result}")
if result != -1:
    print(f"匹配的子串: {text1[result:result+len(pattern1)]}")

## 方法3: 支持查找所有匹配的KMP算法

### 扩展功能

标准KMP只返回第一个匹配位置，但在实际应用中，我们常常需要：
- 查找所有匹配位置
- 统计匹配次数
- 支持重叠匹配

**实现要点**：找到一个匹配后，不立即返回，而是利用next数组继续查找

### 时间复杂度

仍然是 O(m+n)，其中 m 是文本串长度，n 是模式串长度

In [ ]:
def kmp_search_all(text, pattern):
    """
    查找所有匹配位置的KMP算法
    
    参数:
        text: 文本串
        pattern: 模式串
    
    返回:
        所有匹配位置的列表
    """
    n = len(text)
    m = len(pattern)
    
    if m == 0:
        return []
    
    # 构建next数组
    next_arr = get_next(pattern)
    
    matches = []  # 存储所有匹配位置
    j = 0
    
    for i in range(n):
        while j > 0 and text[i] != pattern[j]:
            j = next_arr[j - 1]
        
        if text[i] == pattern[j]:
            j += 1
        
        if j == m:
            # 找到一个匹配，记录位置
            matches.append(i - m + 1)
            # 继续查找下一个匹配（支持重叠）
            j = next_arr[j - 1]
    
    return matches


# 测试查找所有匹配
print("查找所有匹配的KMP算法测试:")
text3 = "ABABABABCABABABAB"
pattern3 = "ABAB"

print(f"文本串: {text3}")
print(f"模式串: {pattern3}")

all_matches = kmp_search_all(text3, pattern3)
print(f"\n所有匹配位置: {all_matches}")
print(f"匹配次数: {len(all_matches)}")

# 显示每个匹配
print("\n匹配详情:")
for idx, pos in enumerate(all_matches):
    substring = text3[pos:pos+len(pattern3)]
    print(f"  匹配{idx+1}: 位置 {pos}, 子串 '{substring}'")
    # 可视化显示匹配位置
    visual = ' ' * pos + '^' * len(pattern3)
    if idx == 0:
        print(f"  文本串: {text3}")
    print(f"          {visual}")

## 方法4: 面向对象的KMP实现

### 设计思路

将KMP算法封装成一个类，提供更灵活的接口：
- 可以查找所有匹配位置
- 可以统计匹配次数
- 可以重复使用next数组

In [ ]:
class KMP:
    """
    KMP字符串匹配算法的面向对象实现
    """
    
    def __init__(self, pattern):
        """
        初始化KMP对象
        
        参数:
            pattern: 模式串
        """
        self.pattern = pattern
        self.next = self._build_next()
    
    def _build_next(self):
        """
        构建next数组（私有方法）
        """
        m = len(self.pattern)
        next_arr = [0] * m
        j = 0
        
        for i in range(1, m):
            while j > 0 and self.pattern[i] != self.pattern[j]:
                j = next_arr[j - 1]
            
            if self.pattern[i] == self.pattern[j]:
                j += 1
            
            next_arr[i] = j
        
        return next_arr
    
    def search(self, text):
        """
        查找第一个匹配位置
        
        参数:
            text: 文本串
        
        返回:
            第一个匹配位置的索引，未找到返回-1
        """
        n = len(text)
        m = len(self.pattern)
        j = 0
        
        for i in range(n):
            while j > 0 and text[i] != self.pattern[j]:
                j = self.next[j - 1]
            
            if text[i] == self.pattern[j]:
                j += 1
            
            if j == m:
                return i - m + 1
        
        return -1
    
    def search_all(self, text):
        """
        查找所有匹配位置
        
        参数:
            text: 文本串
        
        返回:
            所有匹配位置的列表
        """
        n = len(text)
        m = len(self.pattern)
        matches = []
        j = 0
        
        for i in range(n):
            while j > 0 and text[i] != self.pattern[j]:
                j = self.next[j - 1]
            
            if text[i] == self.pattern[j]:
                j += 1
            
            if j == m:
                matches.append(i - m + 1)
                j = self.next[j - 1]  # 继续查找下一个匹配
        
        return matches
    
    def count(self, text):
        """
        统计匹配次数
        
        参数:
            text: 文本串
        
        返回:
            匹配次数
        """
        return len(self.search_all(text))
    
    def get_next_array(self):
        """
        获取next数组
        """
        return self.next.copy()


# 测试面向对象实现
print("面向对象KMP实现测试:")
text3 = "ABABABABCABABABABCABAB"
pattern3 = "ABAB"

kmp = KMP(pattern3)

print(f"文本串: {text3}")
print(f"模式串: {pattern3}")
print(f"next数组: {kmp.get_next_array()}")

# 查找第一个匹配
first_match = kmp.search(text3)
print(f"\n第一个匹配位置: {first_match}")

# 查找所有匹配
all_matches = kmp.search_all(text3)
print(f"所有匹配位置: {all_matches}")

# 统计匹配次数
count = kmp.count(text3)
print(f"匹配次数: {count}")

# 显示每个匹配
print("\n匹配详情:")
for idx, pos in enumerate(all_matches):
    print(f"  匹配{idx+1}: 位置{pos}, 子串='{text3[pos:pos+len(pattern3)]}'")

## 方法5: 简洁版KMP实现

### 特点

- 代码简洁，易于理解和记忆
- 适合面试和竞赛快速实现
- 功能完整

In [ ]:
def kmp(text, pattern):
    """
    简洁版KMP实现
    """
    # 构建next数组
    def build_next(p):
        next_arr = [0] * len(p)
        j = 0
        for i in range(1, len(p)):
            while j > 0 and p[i] != p[j]:
                j = next_arr[j - 1]
            if p[i] == p[j]:
                j += 1
            next_arr[i] = j
        return next_arr
    
    # 执行匹配
    next_arr = build_next(pattern)
    j = 0
    for i in range(len(text)):
        while j > 0 and text[i] != pattern[j]:
            j = next_arr[j - 1]
        if text[i] == pattern[j]:
            j += 1
        if j == len(pattern):
            return i - len(pattern) + 1
    return -1


# 测试简洁版
print("简洁版KMP测试:")
result = kmp(text1, pattern1)
print(f"文本串: {text1}")
print(f"模式串: {pattern1}")
print(f"匹配位置: {result}")
if result != -1:
    print(f"匹配的子串: {text1[result:result+len(pattern1)]}")

## 综合测试与性能比较

### 测试不同场景下各算法的表现

In [ ]:
import time

# 准备测试数据
test_cases = [
    {
        "name": "基本测试",
        "text": "ABABDABACDABABCABAB",
        "pattern": "ABABCABAB"
    },
    {
        "name": "重复字符测试",
        "text": "AAAAAAAAAB",
        "pattern": "AAAAB"
    },
    {
        "name": "未找到测试",
        "text": "ABCDEFGHIJK",
        "pattern": "XYZ"
    },
    {
        "name": "长文本测试",
        "text": "A" * 1000 + "ABCDEFG" + "B" * 1000,
        "pattern": "ABCDEFG"
    }
]

# 测试函数
algorithms = [
    ("朴素算法", brute_force_search),
    ("标准KMP", kmp_search),
    ("简洁KMP", kmp)
]

print("="*80)
print("算法性能测试")
print("="*80)

for test in test_cases:
    print(f"\n【{test['name']}】")
    print(f"文本串长度: {len(test['text'])}, 模式串长度: {len(test['pattern'])}")
    
    results = []
    for name, func in algorithms:
        start = time.time()
        result = func(test['text'], test['pattern'])
        elapsed = (time.time() - start) * 1000000  # 转换为微秒
        results.append((name, result, elapsed))
    
    # 验证所有算法结果一致
    expected = results[0][1]
    all_correct = all(r[1] == expected for r in results)
    
    print(f"匹配位置: {expected}")
    print(f"结果一致性: {'✓ 通过' if all_correct else '✗ 失败'}")
    print("\n算法性能:")
    for name, result, elapsed in results:
        print(f"  {name:12s}: {elapsed:8.2f} μs")

## KMP算法的应用场景

### 1. 字符串查找
- 文本编辑器中的查找功能
- 搜索引擎中的关键词匹配

### 2. DNA序列分析
- 基因序列比对
- 模式识别

### 3. 网络安全
- 入侵检测系统中的模式匹配
- 病毒特征码匹配

### 4. 数据压缩
- LZ系列压缩算法中的重复序列查找

## 练习题

### 题目1: 实现查找所有匹配位置的函数

In [ ]:
def kmp_find_all(text, pattern):
    """
    使用KMP算法查找所有匹配位置
    
    练习：请补充完整这个函数
    """
    # TODO: 实现查找所有匹配位置的逻辑
    pass

# 测试代码（取消注释来测试）
# text = "ABABABABCABAB"
# pattern = "ABAB"
# print(f"所有匹配位置: {kmp_find_all(text, pattern)}")
# 期望输出: [0, 2, 4, 9]

### 题目2: 统计模式串在文本串中出现的次数（允许重叠）

In [ ]:
def count_pattern(text, pattern):
    """
    统计模式串在文本串中出现的次数
    
    练习：使用KMP算法实现
    """
    # TODO: 实现计数逻辑
    pass

# 测试代码（取消注释来测试）
# text = "ABABABABAB"
# pattern = "ABA"
# print(f"出现次数: {count_pattern(text, pattern)}")
# 期望输出: 4 (位置0, 2, 4, 6)

## 总结

### KMP算法的优势
1. **高效性**：时间复杂度 O(m+n)，优于朴素算法的 O(m*n)
2. **稳定性**：最坏情况下的性能也有保证
3. **实用性**：广泛应用于各种字符串匹配场景

### 核心要点
1. **next数组**是KMP算法的关键，记录了模式串的前缀信息
2. **避免回退**：利用已匹配的信息，减少不必要的比较
3. **两个阶段**：预处理（构建next数组）+ 匹配

### 学习建议
1. 理解next数组的构建过程和含义
2. 手动模拟几个例子，加深理解
3. 对比朴素算法，体会KMP的优化思想
4. 多做练习，熟练掌握实现方法

### 扩展阅读
- Boyer-Moore算法：另一种高效的字符串匹配算法
- Aho-Corasick算法：多模式串匹配
- Rabin-Karp算法：基于哈希的字符串匹配